In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [88]:
train_identity=pd.read_csv('ieee-fraud-detection/train_identity.csv')
train_transaction=pd.read_csv('ieee-fraud-detection/train_transaction.csv')

In [26]:
## ! pip install xgboost

In [27]:
import sklearn
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

In [90]:
transaction_cols = [
    "TransactionID",
    "isFraud",
    "TransactionDT",
    "TransactionAmt",
    "ProductCD",

    # Payment/card information
    "card1", "card2", "card3", "card4", "card5", "card6",

    # Address information
    "addr1", "addr2",

    # Email information
    "P_emaildomain",
    "R_emaildomain",

    # High-signal frequency counters & timedeltas
    "C1", "C2", "C13", "C14",
    "D1", "D2", "D15",
]

identity_cols = [
    "TransactionID",
    "DeviceType",
    "DeviceInfo",
] + [f"id_{i:02d}" for i in range(1, 39)]

tx = train_transaction[transaction_cols].copy()

identity = train_identity[identity_cols].copy()

df = tx.merge(
    identity,
    on="TransactionID",
    how="left"
)


In [91]:
df = df.sort_values("TransactionDT").reset_index(drop=True)

In [89]:
# ============================================================
# 1. CARD-LEVEL CAUSAL MEMORY (Expanding Window)
# ============================================================

# Number of transactions by this card STRICTLY BEFORE current row
df["card_prior_txs"] = df.groupby("card1").cumcount()

# Number of frauds by this card STRICTLY BEFORE current row
# (cumsum including current row minus current row's isFraud)
df["card_prior_frauds"] = (
    df.groupby("card1")["isFraud"].cumsum() - df["isFraud"]
)

# Bayesian Smoothed Prior Fraud Rate (prevents division by zero)
# Using empirical Bayes with global prior ~3.5%
GLOBAL_PRIOR = 0.035
SMOOTH_WEIGHT = 10.0  # weight of prior

df["card_prior_fraud_rate"] = (
    df["card_prior_frauds"] + (SMOOTH_WEIGHT * GLOBAL_PRIOR)
) / (df["card_prior_txs"] + SMOOTH_WEIGHT)

# Time since this card's last transaction (seconds)
df["card_time_delta"] = (
    df.groupby("card1")["TransactionDT"].diff().fillna(999999)
)


# ============================================================
# 2. NETWORK CONTAGION (Device-Level Prior Fraud)
# ============================================================
# "Has ANY card on this Device committed fraud before this moment?"

mask_device = df["DeviceInfo"].notna()
df["device_prior_frauds"] = 0
df["device_prior_txs"] = 0

df.loc[mask_device, "device_prior_txs"] = df[mask_device].groupby(
    "DeviceInfo"
).cumcount()

df.loc[mask_device, "device_prior_frauds"] = (
    df[mask_device].groupby("DeviceInfo")["isFraud"].cumsum()
    - df.loc[mask_device, "isFraud"]
)

# Flag: Card is clean, but device has prior fraud history (Network Contagion)
df["device_is_tainted"] = (
    (df["device_prior_frauds"] > 0) & (df["card_prior_frauds"] == 0)
).astype("int32")

In [92]:
from sklearn.model_selection import train_test_split
# Sort chronologically by TransactionDT
df = df.sort_values("TransactionDT").reset_index(drop=True)

# 80/20 time-based boundary
split_dt = df["TransactionDT"].quantile(0.80)
train_df = df[df["TransactionDT"] <= split_dt].copy()
val_df   = df[df["TransactionDT"] >  split_dt].copy()

print(f"Train time range: {train_df['TransactionDT'].min()} -> {train_df['TransactionDT'].max()}")
print(f"Val time range:   {val_df['TransactionDT'].min()} -> {val_df['TransactionDT'].max()}")

Train time range: 86400 -> 12192842
Val time range:   12192900 -> 15811131


In [ ]:
# ============================================================
# FEATURE DEFINITIONS & CATEGORICAL ENCODING
# ============================================================

# 1. Legacy minimal features (for historical reference: PR-AUC ~0.2427)
MINIMAL_BASE_FEATURES = [
    "TransactionAmt", "ProductCD",
    "card1", "card2", "card3", "card5", "card6",
    "addr1", "addr2",
    "DeviceInfo", "DeviceType",
    "id_01", "id_02", "id_05", "id_06", "id_19", "id_20", "id_31", "id_33",
    "TransactionDT"
]

# 2. True Tabular Baseline features (Standard IEEE-CIS C & D counting/timedelta signals + email)
TABULAR_BASE_FEATURES = MINIMAL_BASE_FEATURES + [
    "C1", "C2", "C13", "C14",
    "D1", "D2", "D15",
    "P_emaildomain"
]

# Work on clean copies
X_train = train_df[TABULAR_BASE_FEATURES].copy()
X_val   = val_df[TABULAR_BASE_FEATURES].copy()

y_train = train_df["isFraud"].astype(int)
y_val   = val_df["isFraud"].astype(int)

# Robust categorical encoding
categorical_cols = [
    "ProductCD", "card6", "DeviceInfo", "DeviceType", "id_31", "id_33", "P_emaildomain"
]

for col in categorical_cols:
    train_values = X_train[col].fillna("__MISSING__").astype(str)
    categories = pd.Index(train_values.unique())
    mapping = {value: idx for idx, value in enumerate(categories)}

    X_train[col] = train_values.map(mapping).astype("int32")
    X_val[col]   = X_val[col].fillna("__MISSING__").astype(str).map(mapping).fillna(-1).astype("int32")

# Fill remaining NaNs with -999 for tree boosting
X_train = X_train.fillna(-999)
X_val   = X_val.fillna(-999)

print(f"X_train shape: {X_train.shape} | X_val shape: {X_val.shape}")
print(f"Baseline features count: {len(TABULAR_BASE_FEATURES)}")


X_train shape: (472432, 28) | X_val shape: (118108, 28)
Baseline features count: 28


In [33]:
X_train.head()

,TransactionAmt,ProductCD,card1,card2,card3,card5,card6,addr1,addr2,DeviceInfo,DeviceType,id_01,id_02,id_05,id_06,id_19,id_20,id_31,id_33,TransactionDT
0,68.5,0,13926,NaN,150.0,142.0,0,315.0,87.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,86400
1,29.0,0,2755,404.0,150.0,102.0,0,325.0,87.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,86401
2,59.0,0,4663,490.0,150.0,166.0,1,330.0,87.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,86469
3,50.0,0,18132,567.0,150.0,117.0,1,476.0,87.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,86499
4,50.0,1,4497,514.0,150.0,102.0,0,420.0,87.0,1,1,0.0,70787.0,NaN,NaN,542.0,144.0,1,1,86506


In [94]:
# ============================================================
# FIT TABULAR BASELINE XGBOOST
# ============================================================
baseline_xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

baseline_xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

val_pred = baseline_xgb.predict_proba(X_val)[:, 1]
print("\n============================================================")
print("             BASELINE BENCHMARK1                ")
print("============================================================")
print(f"ROC-AUC: {roc_auc_score(y_val, val_pred):.4f}")
print(f"PR-AUC : {average_precision_score(y_val, val_pred):.4f}")

[0]	validation_0-aucpr:0.34650
[100]	validation_0-aucpr:0.44350
[200]	validation_0-aucpr:0.46694
[300]	validation_0-aucpr:0.48036
[399]	validation_0-aucpr:0.49389

             BASELINE BENCHMARK1                
ROC-AUC: 0.8914
PR-AUC : 0.4940


In [ ]:
# ============================================================
# ACTOR-CENTRIC FRAUD GRAPH (WCC & SYNDICATE DETECTION)
# ============================================================

import pandas as pd
import numpy as np
import networkx as nx
import math
from collections import defaultdict
from itertools import combinations

# 1. INFERRED ACTOR IDENTITY (Client Proxy)
ACTOR_COLS = ["card1", "addr1", "addr2", "P_emaildomain"]

for df_split in [train_df, val_df]:
    df_split["ActorID"] = df_split[ACTOR_COLS].fillna("__MISSING__").astype(str).agg("|".join, axis=1)

# 2. SHARED IDENTIFIERS
ENTITY_COLS = ["card1", "DeviceInfo", "id_19", "id_20"]
MAX_ACTORS_PER_ENTITY = 1000  # Prune public hubs (Coffee shop filter)
MIN_SHARED_TYPES = 2         # Must share >= 2 independent types to form syndicate edge

entity_actors_by_type = {}
entity_actor_counts = {}
N_ACTORS = train_df["ActorID"].nunique()

for entity_col in ENTITY_COLS:
    temp = train_df[["ActorID", entity_col]].dropna().drop_duplicates()
    counts = temp.groupby(entity_col)["ActorID"].nunique()
    
    # Exclude massive public hubs (> 500 actors)
    valid_entities = set(counts[counts <= MAX_ACTORS_PER_ENTITY].index)
    temp = temp[temp[entity_col].isin(valid_entities)]
    
    entity_actors = defaultdict(set)
    for entity, group in temp.groupby(entity_col):
        entity_actors[entity] = set(group["ActorID"])
        entity_actor_counts[(entity_col, entity)] = len(group["ActorID"])
        
    entity_actors_by_type[entity_col] = entity_actors
    print(f"{entity_col}: {len(entity_actors)} identifiers kept (Hubs pruned)")

# 3. MULTI-TYPE COLLABORATION EDGES
pair_types = defaultdict(set)
pair_evidence = defaultdict(lambda: defaultdict(set))

for entity_col in ENTITY_COLS:
    entity_actors = entity_actors_by_type[entity_col]
    for entity, actors in entity_actors.items():
        if len(actors) < 2 or len(actors) > 200:
            continue
        for a1, a2 in combinations(sorted(actors), 2):
            pair = (a1, a2)
            pair_types[pair].add(entity_col)
            pair_evidence[pair][entity_col].add(entity)

# Keep only multi-type syndicate links
strong_pairs = {p: types for p, types in pair_types.items() if len(types) >= MIN_SHARED_TYPES}
print(f"\nCandidate pairs: {len(pair_types)} | Strong syndicate links (>=2 types): {len(strong_pairs)}")

# 4. BUILD G_actor WITH EDGE RARITY
G_actor = nx.Graph()
actors = train_df["ActorID"].dropna().unique()
G_actor.add_nodes_from([(a, {"type": "Actor"}) for a in actors])

for (a1, a2), types in strong_pairs.items():
    evidence = {}
    total_rarity = 0.0
    for entity_type in types:
        vals = list(pair_evidence[(a1, a2)][entity_type])
        evidence[entity_type] = vals
        
        # IDF rarity: rare devices get higher score
        best_r = 0.0
        for v in vals:
            cnt = entity_actor_counts.get((entity_type, v), 1)
            r = math.log(N_ACTORS / cnt)
            if r > best_r:
                best_r = r
        total_rarity += best_r

    G_actor.add_edge(
        a1, a2,
        shared_types=len(types),
        shared_entity_types=list(types),
        evidence=evidence,
        rarity_score=float(total_rarity)
    )

print(f"Actor Graph: {G_actor.number_of_nodes():,} actors | {G_actor.number_of_edges():,} clean syndicate edges")

card1: 12730 identifiers kept (Hubs pruned)
DeviceInfo: 1634 identifiers kept (Hubs pruned)
id_19: 496 identifiers kept (Hubs pruned)
id_20: 367 identifiers kept (Hubs pruned)

Candidate pairs: 2040225 | Strong syndicate links (>=2 types): 37269
Actor Graph: 80,961 actors | 37,269 clean syndicate edges


In [100]:
# ============================================================
# EXTRACT TOPOLOGICAL METRICS FROM G_actor
# ============================================================
actor_features = []

for actor in G_actor.nodes():
    neighbors = list(G_actor.neighbors(actor))
    
    if neighbors:
        edge_data = [G_actor[actor][n] for n in neighbors]
        rarities = [d.get("rarity_score", 0.0) for d in edge_data]
        shared_types = [d.get("shared_types", 0) for d in edge_data]
        
        actor_features.append({
            "ActorID": actor,
            "actor_degree": len(neighbors),
            "mean_rarity": np.mean(rarities),
            "max_rarity": np.max(rarities),
            "mean_shared_types": np.mean(shared_types),
            "max_shared_types": np.max(shared_types)
        })
    else:
        actor_features.append({
            "ActorID": actor,
            "actor_degree": 0,
            "mean_rarity": 0.0,
            "max_rarity": 0.0,
            "mean_shared_types": 0.0,
            "max_shared_types": 0.0
        })

graph_features_df = pd.DataFrame(actor_features)
print(f"Extracted features for {len(graph_features_df):,} actors. Shape: {graph_features_df.shape}")
graph_features_df.head()

Extracted features for 80,961 actors. Shape: (80961, 6)


,ActorID,actor_degree,mean_rarity,max_rarity,mean_shared_types,max_shared_types
0,13926|315.0|87.0|__MISSING__,0,0.0,0.0,0.0,0.0
1,2755|325.0|87.0|gmail.com,0,0.0,0.0,0.0,0.0
2,4663|330.0|87.0|outlook.com,0,0.0,0.0,0.0,0.0
3,18132|476.0|87.0|yahoo.com,0,0.0,0.0,0.0,0.0
4,4497|420.0|87.0|gmail.com,0,0.0,0.0,0.0,0.0


In [101]:
# ============================================================
# PATH ALGORITHMS: TARGETED BFS TO KNOWN FRAUD RINGS
# ============================================================

# Known fraudulent actors in training set
known_fraud_actors = set(
    train_df[train_df["isFraud"] == 1]["ActorID"].dropna().unique()
)
print(f"Known fraudulent seed actors: {len(known_fraud_actors):,}")

# 1. Shortest Path Distance to nearest known fraudster (capped at 3 hops)
hop_distances = nx.multi_source_dijkstra_path_length(
    G_actor, sources=known_fraud_actors, cutoff=3
)

# 2. Assign path features to actors
graph_features_df["hop_to_fraud"] = (
    graph_features_df["ActorID"]
    .map(hop_distances)
    .fillna(999)  # 999 = completely disconnected from fraud
    .astype("int32")
)

# 3. Direct Accomplice / Mule Flag (hop == 1 but self is unflagged)
actor_is_fraud = train_df.groupby("ActorID")["isFraud"].max().to_dict()

graph_features_df["is_direct_accomplice"] = graph_features_df["ActorID"].apply(
    lambda a: 1
    if (actor_is_fraud.get(a, 0) == 0 and hop_distances.get(a, 999) == 1)
    else 0
)

print("\nPath algorithm features summary:")
print(graph_features_df["hop_to_fraud"].value_counts())
print(
    f"Unflagged direct accomplices caught: {graph_features_df['is_direct_accomplice'].sum():,}"
)

Known fraudulent seed actors: 4,392

Path algorithm features summary:
hop_to_fraud
999    74671
0       4392
1       1496
2        341
3         61
Name: count, dtype: int64
Unflagged direct accomplices caught: 1,496


In [102]:
# ============================================================
# 1. WEAKLY CONNECTED COMPONENTS (Reachability / Isolated Rings)
# ============================================================
wcc_components = list(nx.connected_components(G_actor))
wcc_map = {}
wcc_size_map = {}

for comp_id, comp in enumerate(wcc_components):
    c_size = len(comp)
    for actor in comp:
        wcc_map[actor] = comp_id
        wcc_size_map[actor] = c_size

graph_features_df["wcc_id"] = graph_features_df["ActorID"].map(wcc_map).fillna(-1).astype("int32")
graph_features_df["wcc_size"] = graph_features_df["ActorID"].map(wcc_size_map).fillna(1).astype("int32")

# ============================================================
# 2. LOUVAIN COMMUNITY DETECTION (Coordinated Syndicate Cliques)
# ============================================================
communities = nx.community.louvain_communities(
    G_actor,
    weight="rarity_score",
    resolution=1.0,
    seed=42
)

actor_to_community = {}
community_size_map = {}

for comm_id, comm in enumerate(communities):
    c_size = len(comm)
    for actor in comm:
        actor_to_community[actor] = comm_id
        community_size_map[actor] = c_size

graph_features_df["community"] = graph_features_df["ActorID"].map(actor_to_community).fillna(-1).astype("int32")
graph_features_df["community_size"] = graph_features_df["ActorID"].map(community_size_map).fillna(1).astype("int32")

print(f"WCC Components: {len(wcc_components):,} | Louvain Communities: {len(communities):,}")

WCC Components: 77,777 | Louvain Communities: 77,823


In [103]:
# ============================================================
# 3. NEO4J GDS FRAUD RING METRICS (Community Contagion)
# ============================================================

# Known fraud label per actor in training data
actor_fraud_status = train_df.groupby("ActorID")["isFraud"].max().to_dict()

# Calculate Community-level summary: userCount, flaggedCount, flaggedPercent
community_stats = []

for comm_id, members in enumerate(communities):
    u_count = len(members)
    f_count = sum(actor_fraud_status.get(m, 0) for m in members)
    f_pct = float(f_count / u_count)
    
    community_stats.append({
        "community": comm_id,
        "userCount": u_count,
        "flaggedCount": f_count,
        "flaggedPercent": round(f_pct, 4)
    })

comm_stats_df = pd.DataFrame(community_stats)

# Merge community metrics
graph_features_df = graph_features_df.merge(comm_stats_df, on="community", how="left")

# 4. UNFLAGGED CONNECTED TO FLAGGED (Guilt by Association / Mule Flag)
mule_flags = {}

for actor in G_actor.nodes():
    is_self_fraud = actor_fraud_status.get(actor, 0)
    # Check if any neighbor is a known fraudster
    has_flagged_neighbor = any(actor_fraud_status.get(nbr, 0) == 1 for nbr in G_actor.neighbors(actor))
    
    # Unflagged user linked to known fraudsters
    mule_flags[actor] = 1 if (is_self_fraud == 0 and has_flagged_neighbor) else 0

graph_features_df["unflagged_connected_to_flagged"] = (
    graph_features_df["ActorID"].map(mule_flags).fillna(0).astype("int32")
)

# Bridge mapping: attach card1 to graph_features_df for seamless downstream merges
actor_to_card = train_df.groupby("ActorID")["card1"].first().to_dict()
graph_features_df["card1"] = graph_features_df["ActorID"].map(actor_to_card)

print("Fraud Ring Metrics Computed Successfully!")
print(graph_features_df[["userCount", "flaggedCount", "flaggedPercent", "unflagged_connected_to_flagged"]].describe())

Fraud Ring Metrics Computed Successfully!
          userCount  flaggedCount  flaggedPercent  \
count  80961.000000  80961.000000    80961.000000   
mean      13.936821      5.604871        0.054248   
std       93.351573     43.829953        0.213028   
min        1.000000      0.000000        0.000000   
25%        1.000000      0.000000        0.000000   
50%        1.000000      0.000000        0.000000   
75%        1.000000      0.000000        0.000000   
max      844.000000    407.000000        1.000000   

       unflagged_connected_to_flagged  
count                    80961.000000  
mean                         0.018478  
std                          0.134673  
min                          0.000000  
25%                          0.000000  
50%                          0.000000  
75%                          0.000000  
max                          1.000000  


In [104]:
# ============================================================
# MERGE FRAUD RING FEATURES INTO TRAIN AND VAL
# ============================================================
FEATURE_COLS = [
    "ActorID",
    "actor_degree",
    "mean_rarity",
    "max_rarity",
    "mean_shared_types",
    "max_shared_types",
    "wcc_size",
    "community_size",
    "userCount",
    "flaggedPercent",
    "unflagged_connected_to_flagged"
]

train_enhanced = train_df.merge(graph_features_df[FEATURE_COLS], on="ActorID", how="left").fillna(0)
val_enhanced   = val_df.merge(graph_features_df[FEATURE_COLS], on="ActorID", how="left").fillna(0)

print("Train enhanced shape:", train_enhanced.shape)
print("Val enhanced shape:  ", val_enhanced.shape)

Train enhanced shape: (472432, 73)
Val enhanced shape:   (118108, 73)


In [105]:
train_enhanced.head()
print(train_enhanced.shape)

(472432, 73)


In [43]:
train_df.shape

(472432, 63)

In [44]:
train_enhanced.isna().any().any()

np.False_

In [45]:
val_enhanced.head()
val_enhanced.shape

(118108, 73)

In [106]:
# ============================================================
# ACTOR GRAPH FEATURES FOR BASELINE COMPARISON
# ============================================================

GRAPH_FEATURES = [
    "actor_degree",
    "mean_rarity",
    "max_rarity",
    "mean_shared_types",
    "max_shared_types",
    "wcc_size",
    "community_size",
    "userCount",
    "flaggedPercent",
    "unflagged_connected_to_flagged",
]

MODEL_FEATURES = MINIMAL_BASE_FEATURES + GRAPH_FEATURES

X_train_graph = train_enhanced[MODEL_FEATURES].copy()
X_val_graph = val_enhanced[MODEL_FEATURES].copy()

y_train = train_enhanced["isFraud"].astype(int)
y_val = val_enhanced["isFraud"].astype(int)

print("Total model features:", len(MODEL_FEATURES))
print(MODEL_FEATURES)

Total model features: 30
['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card5', 'card6', 'addr1', 'addr2', 'DeviceInfo', 'DeviceType', 'id_01', 'id_02', 'id_05', 'id_06', 'id_19', 'id_20', 'id_31', 'id_33', 'TransactionDT', 'actor_degree', 'mean_rarity', 'max_rarity', 'mean_shared_types', 'max_shared_types', 'wcc_size', 'community_size', 'userCount', 'flaggedPercent', 'unflagged_connected_to_flagged']


In [107]:
categorical_cols = [
    "ProductCD",
    "card6",
    "DeviceInfo",
    "DeviceType",
    "id_31",
    "id_33"
]

for col in categorical_cols:

    combined = pd.concat([
        X_train_graph[col],
        X_val_graph[col]
    ]).fillna("__MISSING__").astype(str)

    categories = pd.Index(combined.unique())

    mapping = {
        value: idx
        for idx, value in enumerate(categories)
    }

    X_train_graph[col] = (
        X_train_graph[col]
        .fillna("__MISSING__")
        .astype(str)
        .map(mapping)
        .astype("int32")
    )

    X_val_graph[col] = (
        X_val_graph[col]
        .fillna("__MISSING__")
        .astype(str)
        .map(mapping)
        .fillna(-1)
        .astype("int32")
    )

In [108]:
graph_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",
    eval_metric="aucpr",

    scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),

    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

graph_model.fit(
    X_train_graph,
    y_train,
    eval_set=[(X_val_graph, y_val)],
    verbose=50
)

[0]	validation_0-aucpr:0.12190
[50]	validation_0-aucpr:0.17047
[100]	validation_0-aucpr:0.17828
[150]	validation_0-aucpr:0.18598
[200]	validation_0-aucpr:0.19812
[250]	validation_0-aucpr:0.19992
[300]	validation_0-aucpr:0.20821
[350]	validation_0-aucpr:0.20810
[400]	validation_0-aucpr:0.21165
[450]	validation_0-aucpr:0.21015
[499]	validation_0-aucpr:0.21543


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [109]:
graph_pred = graph_model.predict_proba(
    X_val_graph
)[:, 1]

graph_roc = roc_auc_score(
    y_val,
    graph_pred
)

graph_pr = average_precision_score(
    y_val,
    graph_pred
)

print("\n========== GRAPH XGBOOST ==========")
print("ROC-AUC:", graph_roc)
print("PR-AUC :", graph_pr)


========== GRAPH XGBOOST ==========
ROC-AUC: 0.7803188889987067
PR-AUC : 0.21567323031684577


In [110]:
importance_df = pd.DataFrame({
    "feature": X_train_graph.columns,
    "importance": graph_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(
    importance_df.head(20).to_string(index=False)
)

                       feature  importance
                flaggedPercent    0.475352
                         addr2    0.165407
unflagged_connected_to_flagged    0.088587
                         card6    0.049855
                    DeviceType    0.023862
                     ProductCD    0.022496
                         card3    0.014713
                         id_01    0.011438
                      wcc_size    0.010459
                  actor_degree    0.009289
                         card2    0.009104
                TransactionAmt    0.009082
                         id_19    0.008467
                         card5    0.007824
                    max_rarity    0.007530
                 TransactionDT    0.007339
                         card1    0.007240
                         id_20    0.007090
                         addr1    0.006708
                         id_02    0.006608


In [111]:
print(
    "\n========== GRAPH FEATURE IMPORTANCE ==========\n"
)

print(
    importance_df[
        importance_df["feature"].isin(GRAPH_FEATURES)
    ].to_string(index=False)
)


========== GRAPH FEATURE IMPORTANCE ==========

                       feature  importance
                flaggedPercent    0.475352
unflagged_connected_to_flagged    0.088587
                      wcc_size    0.010459
                  actor_degree    0.009289
                    max_rarity    0.007530
                   mean_rarity    0.006158
                community_size    0.005403
             mean_shared_types    0.005156
                     userCount    0.004217
              max_shared_types    0.003444


In [112]:
from sklearn.metrics import precision_score

for pct in [0.1, 0.5, 1, 2, 5]:

    n = int(len(y_val) * pct / 100)

    idx = np.argsort(graph_pred)[::-1][:n]

    precision = y_val.iloc[idx].mean()

    print(
        f"Top {pct}% precision: {precision:.4f}"
    )

Top 0.1% precision: 0.6525
Top 0.5% precision: 0.5153
Top 1% precision: 0.4454
Top 2% precision: 0.3480
Top 5% precision: 0.2469


In [55]:
for pct in [0.1, 0.5, 1, 2, 5]:

    n = int(len(y_val) * pct / 100)

    idx = np.argsort(val_pred)[::-1][:n]

    precision = y_val.iloc[idx].mean()

    print(
        f"Baseline top {pct}% precision: {precision:.4f}"
    )

Baseline top 0.1% precision: 0.7712
Baseline top 0.5% precision: 0.5949
Baseline top 1% precision: 0.5021
Baseline top 2% precision: 0.3916
Baseline top 5% precision: 0.2437


## Our Model 2 -> Graph Neutal Network

In [56]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import RGCNConv


c:\Users\Anirban Banerjee\anaconda3\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Anirban Banerjee\anaconda3\envs\myenv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [113]:
# ============================================================
# VERIFY EDGE RARITY ON G_actor
# ============================================================

rarity_values = [d.get("rarity_score", 0.0) for _, _, d in G_actor.edges(data=True)]

print(f"Edges with rarity score: {len(rarity_values):,}")
print(f"Mean edge rarity: {np.mean(rarity_values):.4f}")
print(f"Max edge rarity:  {np.max(rarity_values):.4f}")



Edges with rarity score: 37,269
Mean edge rarity: 15.5187
Max edge rarity:  35.7712


In [114]:
import numpy as np
import pandas as pd
import networkx as nx
from collections import defaultdict


# ============================================================
# 1. TRANSACTION TIME INDEX
# ============================================================

fraud_tx = train_df[train_df["isFraud"] == 1].copy()

fraud_times = (
    fraud_tx
    .groupby("card1")["TransactionDT"]
    .apply(lambda x: np.sort(x.values))
    .to_dict()
)

fraud_card_set = set(fraud_times.keys())

print("Fraudulent cards:", len(fraud_card_set))


# ============================================================
# 2. TEMPORAL COORDINATION
# ============================================================

def temporal_proximity(times_a, times_b, window=86400):
    """
    Fraction of transactions from A that occur within
    `window` seconds of at least one transaction from B.
    """

    if len(times_a) == 0 or len(times_b) == 0:
        return 0.0

    idx = np.searchsorted(times_b, times_a)

    nearest = np.full(len(times_a), np.inf)

    # Right neighbour
    valid = idx < len(times_b)
    nearest[valid] = np.abs(
        times_b[idx[valid]] - times_a[valid]
    )

    # Left neighbour
    valid_left = idx > 0
    nearest[valid_left] = np.minimum(
        nearest[valid_left],
        np.abs(
            times_b[idx[valid_left] - 1] -
            times_a[valid_left]
        )
    )

    return np.mean(nearest <= window)


def symmetric_temporal_proximity(times_a, times_b, window=86400):

    ab = temporal_proximity(times_a, times_b, window)
    ba = temporal_proximity(times_b, times_a, window)

    return (ab + ba) / 2

Fraudulent cards: 1532


In [115]:
# ============================================================
# ACTOR EDGE TEMPORAL COORDINATION
# ============================================================

# All transaction timestamps grouped by ActorID
actor_times = (
    train_df.groupby("ActorID")["TransactionDT"]
    .apply(lambda x: np.sort(x.values))
    .to_dict()
)

print(f"Actors with timestamps: {len(actor_times):,}")

# Add temporal coordination to every G_actor edge
for u, v, data in G_actor.edges(data=True):
  times_u = actor_times.get(u, np.array([]))
  times_v = actor_times.get(v, np.array([]))

  # Uses symmetric_temporal_proximity from Cell 35
  data["temporal"] = float(
      symmetric_temporal_proximity(times_u, times_v, window=86400)
  )

temporal_vals = [d["temporal"] for _, _, d in G_actor.edges(data=True)]
print(f"Temporal features added to G_actor edges.")
print(f"Temporal mean: {np.mean(temporal_vals):.4f}")
print(f"Temporal max:  {np.max(temporal_vals):.4f}")

Actors with timestamps: 80,961
Temporal features added to G_actor edges.
Temporal mean: 0.4008
Temporal max:  1.0000


In [116]:
card_stats = (
    train_df.groupby("card1")
      .agg(
          transactions=("TransactionID", "count"),
          frauds=("isFraud", "sum")
      )
)

card_stats["fraud_rate"] = (
    card_stats["frauds"] /
    card_stats["transactions"]
)

In [61]:
card_stats[card_stats["fraud_rate"]>0]

,transactions,frauds,fraud_rate
card1,,,
1015,8,1,0.125000
1016,12,1,0.083333
1022,7,1,0.142857
1043,4,2,0.500000
1047,16,4,0.250000
...,...,...,...
18343,189,1,0.005291
18366,402,6,0.014925
18370,158,11,0.069620


In [118]:
# ============================================================
# ACTOR GNN NODE FEATURE MATRIX 
# ============================================================

actors = list(G_actor.nodes())
actor_to_idx = {actor: i for i, actor in enumerate(actors)}
idx_to_actor = {i: actor for actor, i in actor_to_idx.items()}

# 1. Combine train_df and val_df (both are guaranteed to have ActorID)
all_tx = pd.concat([train_df, val_df], axis=0, ignore_index=True)

# 2. Pre-aggregate transaction volume, spending, and entity diversity (Strictly Non-Target!)
actor_tx_stats = (
    all_tx.groupby("ActorID")
    .agg(
        tx_count=("TransactionID", "count"),
        amt_mean=("TransactionAmt", "mean"),
        amt_max=("TransactionAmt", "max"),
        amt_std=("TransactionAmt", "std"),
        device_count=("DeviceInfo", "nunique"),
        card_count=("card1", "nunique")
    )
    .fillna(0)
    .reset_index()
)

# 3. Merge structural topology & spending stats onto actors
actor_profile = (
    pd.DataFrame({"ActorID": actors})
    .merge(graph_features_df[["ActorID", "actor_degree", "wcc_size", "community_size", "mean_rarity"]], on="ActorID", how="left")
    .merge(actor_tx_stats, on="ActorID", how="left")
    .fillna(0)
)

# 10 Pure Structural & Behavioral Node Features (ZERO TARGET LEAKAGE):
GNN_NODE_COLS = [
    "actor_degree",       # Degree centrality in ER graph
    "wcc_size",           # Connected component size
    "community_size",     # Louvain community size
    "tx_count",           # Total activity volume
    "amt_mean",           # Spending mean
    "amt_max",            # Peak transaction size
    "amt_std",            # Spending volatility
    "device_count",       # Distinct devices shared/used
    "card_count",         # Distinct payment cards
    "mean_rarity",        # Average rarity score of shared entities
]

X = actor_profile[GNN_NODE_COLS].values
print(f"Actor GNN Node Feature Matrix X shape: {X.shape} (10 pure non-target features)")

# 4. Ground Truth Labels & Train Mask for Supervised Actor Fraud Head
train_actor_set = set(train_df["ActorID"].unique())
actor_train_frauds = train_df.groupby("ActorID")["isFraud"].max().to_dict()

# Binary mask: True if actor is in training set
train_actor_mask = np.array([a in train_actor_set for a in actors], dtype=bool)
# Target label: 1 if actor committed fraud in train_df, else 0
actor_train_labels = np.array([actor_train_frauds.get(a, 0) for a in actors], dtype=np.float32)

print(f"Total Actors: {len(actors):,} | Train Actors: {train_actor_mask.sum():,} | Val-only Actors: {(~train_actor_mask).sum():,}")
print(f"Known Fraudulent Train Actors: {int(actor_train_labels[train_actor_mask].sum()):,}")

Actor GNN Node Feature Matrix X shape: (80961, 10) (10 pure non-target features)
Total Actors: 80,961 | Train Actors: 80,961 | Val-only Actors: 0
Known Fraudulent Train Actors: 4,392


In [119]:
# ============================================================
# EDGE INDEX + EDGE FEATURES
# ============================================================

# ============================================================
# ACTOR GRAPH EDGE TENSORS (EVIDENCE + RARITY + TEMPORAL)
# ============================================================

edge_index = []
edge_attr = []

for u, v, data in G_actor.edges(data=True):
  u_idx = actor_to_idx[u]
  v_idx = actor_to_idx[v]

  shared_t = float(data.get("shared_types", 1))
  rarity_s = float(data.get("rarity_score", 1.0))
  temporal_s = float(data.get("temporal", 0.0))

  # Undirected: add both directions
  edge_index.append([u_idx, v_idx])
  edge_attr.append([shared_t, rarity_s, temporal_s])

  edge_index.append([v_idx, u_idx])
  edge_attr.append([shared_t, rarity_s, temporal_s])

print(
    f"Actor Graph Edge Index: {len(edge_index):,} edges with 3 attributes"
    " [shared_types, rarity, temporal]"
)

Actor Graph Edge Index: 74,538 edges with 3 attributes [shared_types, rarity, temporal]


In [120]:
# ============================================================
# HOP NEIGHBOR INDEX
# ============================================================

from collections import defaultdict
import numpy as np

num_nodes = len(actors)

# ------------------------------------------------------------
# 1-HOP NEIGHBORS
# ------------------------------------------------------------

neighbors_1hop = [[] for _ in range(num_nodes)]

for src, dst in edge_index:
    neighbors_1hop[src].append(dst)


# Convert to numpy arrays
neighbors_1hop = [
    np.asarray(n, dtype=np.int64)
    for n in neighbors_1hop
]


print(f"Adjacency list ready for {num_nodes:,} actors")

degrees = np.array([
    len(n)
    for n in neighbors_1hop
])

print(f"Mean degree: {degrees.mean():.2f}")
print(f"Median degree: {np.median(degrees):.2f}")
print(f"Max degree: {degrees.max():.2f}")

Adjacency list ready for 80,961 actors
Mean degree: 0.92
Median degree: 0.00
Max degree: 742.00


In [121]:
# ============================================================
# HOP SAMPLER
# ============================================================

def sample_neighbors(
    node_ids,
    fanout_1=20,
    fanout_2=10,
    rng=None
):
    """
    Sample:
        target
          ↓
        1-hop
          ↓
        2-hop

    Returns:
        target_nodes
        hop1_nodes
        hop2_nodes
    """

    if rng is None:
        rng = np.random.default_rng(42)

    node_ids = np.asarray(
        node_ids,
        dtype=np.int64
    )

    hop1 = set()
    hop2 = set()

    # --------------------------------------------------------
    # 1-HOP
    # --------------------------------------------------------

    for node in node_ids:

        nbrs = neighbors_1hop[node]

        if len(nbrs) == 0:
            continue

        if len(nbrs) > fanout_1:
            sampled = rng.choice(
                nbrs,
                size=fanout_1,
                replace=False
            )
        else:
            sampled = nbrs

        hop1.update(sampled.tolist())


    # --------------------------------------------------------
    # 2-HOP
    # --------------------------------------------------------

    for node in hop1:

        nbrs = neighbors_1hop[node]

        if len(nbrs) == 0:
            continue

        if len(nbrs) > fanout_2:
            sampled = rng.choice(
                nbrs,
                size=fanout_2,
                replace=False
            )
        else:
            sampled = nbrs

        hop2.update(sampled.tolist())


    # Don't duplicate target nodes
    hop1.difference_update(node_ids.tolist())
    hop2.difference_update(node_ids.tolist())
    hop2.difference_update(hop1)

    return (
        node_ids,
        np.asarray(list(hop1), dtype=np.int64),
        np.asarray(list(hop2), dtype=np.int64)
    )

In [66]:
# ============================================================
# TEST SAMPLER
# ============================================================

rng = np.random.default_rng(42)

test_nodes = rng.choice(
    num_nodes,
    size=10,
    replace=False
)

target, hop1, hop2 = sample_neighbors(
    test_nodes,
    fanout_1=20,
    fanout_2=10,
    rng=rng
)

print("Target nodes:", len(target))
print("1-hop nodes:", len(hop1))
print("2-hop nodes:", len(hop2))

Target nodes: 10
1-hop nodes: 0
2-hop nodes: 0


In [122]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# MULTI-TASK ACTOR GRAPH ENCODER ARCHITECTURE
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentiveEdgeConv(nn.Module):
    def __init__(self, in_dim, edge_dim, out_dim):
        super().__init__()
        self.node_lin = nn.Linear(in_dim, out_dim)
        self.edge_lin = nn.Linear(edge_dim, out_dim)
        self.self_lin = nn.Linear(in_dim, out_dim)
        self.att_node = nn.Linear(in_dim, 1)
        self.att_edge = nn.Linear(edge_dim, 1)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x, edge_index, edge_attr):
        row, col = edge_index
        h_node_j = self.node_lin(x[col])
        h_edge = self.edge_lin(edge_attr)
        a_node_j = self.att_node(x[col])
        a_edge = self.att_edge(edge_attr)

        msg = self.act(h_node_j + h_edge)
        alpha = torch.sigmoid(a_node_j + a_edge)
        weighted_msg = alpha * msg

        out = torch.zeros(x.size(0), h_node_j.size(1), device=x.device)
        out.index_add_(0, row, weighted_msg)
        return F.relu(self.self_lin(x) + out)


class MultiTaskActorGNN(nn.Module):
    def __init__(self, in_dim=10, edge_dim=3, hidden_dim=48, out_dim=32):
        super().__init__()
        # 2-Layer Attentive Message Passing Backbone
        self.conv1 = AttentiveEdgeConv(in_dim, edge_dim, hidden_dim)
        self.conv2 = AttentiveEdgeConv(hidden_dim, edge_dim, out_dim)

        # 1. Supervised Actor Fraud Head (Predicts actor fraud probability)
        self.actor_fraud_head = nn.Sequential(
            nn.Linear(out_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1)
        )

        # 2. Ring / Edge Link Affinity Head (Distinguishes true high-rarity sharing from non-edges)
        self.edge_ring_head = nn.Sequential(
            nn.Linear(out_dim * 2 + edge_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

        # 3. Contrastive Projection Head (InfoNCE Auxiliary Regularization)
        self.proj_head = nn.Sequential(
            nn.Linear(out_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )

    def forward(self, x, edge_index, edge_attr):
        h1 = self.conv1(x, edge_index, edge_attr)
        h1 = F.dropout(h1, p=0.2, training=self.training)
        h2 = self.conv2(h1, edge_index, edge_attr)
        return h2

    def predict_actor_fraud(self, h):
        return self.actor_fraud_head(h).squeeze(-1)

    def predict_edge(self, h_u, h_v, edge_attr):
        pair = torch.cat([h_u, h_v, edge_attr], dim=-1)
        return self.edge_ring_head(pair).squeeze(-1)

    def project(self, h):
        return self.proj_head(h)

In [123]:
# ============================================================
# CELL 123 (REPLACEMENT): PREPARE TENSORS
# ============================================================
from sklearn.preprocessing import StandardScaler

node_scaler = StandardScaler()
X_norm = node_scaler.fit_transform(X)

edge_scaler = StandardScaler()
E_norm = edge_scaler.fit_transform(edge_attr)

x_tensor = torch.tensor(X_norm, dtype=torch.float32)
edge_attr_tensor = torch.tensor(E_norm, dtype=torch.float32)
edge_index_tensor = torch.tensor(np.asarray(edge_index).T, dtype=torch.long)

print("x_tensor:", x_tensor.shape)
print("edge_index_tensor:", edge_index_tensor.shape)
print("edge_attr_tensor:", edge_attr_tensor.shape)

x_tensor: torch.Size([80961, 10])
edge_index_tensor: torch.Size([2, 74538])
edge_attr_tensor: torch.Size([74538, 3])


In [124]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

# Node IDs as seeds
seed_dataset = TensorDataset(torch.arange(len(actors), dtype=torch.long))
train_seeds_loader = DataLoader(seed_dataset, batch_size=1024, shuffle=True)

def fetch_subgraph(seed_batch_nodes, fanout_1=15, fanout_2=10):
    # Uses your sample_neighbors function from Cell [148]
    targets, hop1, hop2 = sample_neighbors(
        seed_batch_nodes.numpy(),
        fanout_1=fanout_1,
        fanout_2=fanout_2
    )
    
    # Combined subnodes (seeds come first)
    all_nodes = np.concatenate([targets, hop1, hop2])
    unique_nodes, inv_idx = np.unique(all_nodes, return_inverse=True)
    
    # Map global card indices to local 0..N indices
    node_map = {n: i for i, n in enumerate(unique_nodes)}
    node_set = set(unique_nodes)
    
    sub_edges = []
    sub_edge_attrs = []
    
    for u in unique_nodes:
        nbrs = neighbors_1hop[u]
        for v in nbrs:
            if v in node_set:
                sub_edges.append([node_map[u], node_map[v]])
                u_actor = idx_to_actor[u]
                v_actor = idx_to_actor[v]
                edge_data = G_actor[u_actor][v_actor]
                sub_edge_attrs.append([
                    float(edge_data.get("shared_types", 1)),
                    float(edge_data.get("rarity_score", 1.0)),
                    float(edge_data.get("temporal", 0.0))
                ])
                
    sub_x = torch.tensor(X_norm[unique_nodes], dtype=torch.float32)
    sub_edge_index = (
        torch.tensor(sub_edges, dtype=torch.long).T 
        if sub_edges else torch.empty((2, 0), dtype=torch.long)
    )
    sub_edge_attr = (
        torch.tensor(sub_edge_attrs, dtype=torch.float32) 
        if sub_edge_attrs else torch.empty((0, 3), dtype=torch.float32)
    )
    
    return sub_x, sub_edge_index, sub_edge_attr, len(targets)

print(f"Total batches per epoch: {len(train_seeds_loader)}")

Total batches per epoch: 80


In [125]:
# ============================================================
# MULTI-TASK GNN TRAINING LOOP (FRAUD BCE + EDGE LOSS + INFONCE)
# ============================================================

import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Multi-Task GNN on: {device}")

def augment_graph(x, edge_index, edge_attr, drop_edge_p=0.2, mask_feat_p=0.2):
    # 1. Feature Masking
    mask = torch.rand(x.shape, device=x.device) > mask_feat_p
    x_aug = x * mask

    # 2. Edge Drop
    num_edges = edge_index.shape[1]
    keep_mask = torch.rand(num_edges, device=edge_index.device) > drop_edge_p
    edge_index_aug = edge_index[:, keep_mask]
    edge_attr_aug = edge_attr[keep_mask]

    return x_aug, edge_index_aug, edge_attr_aug



model = MultiTaskActorGNN(
    in_dim=10, edge_dim=3, hidden_dim=48, out_dim=32
).to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

train_pos_count = actor_train_labels[train_actor_mask].sum()
train_neg_count = train_actor_mask.sum() - train_pos_count
pos_weight_val = float(train_neg_count / max(1.0, train_pos_count))
fraud_bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val, device=device))
edge_bce_loss_fn = nn.BCEWithLogitsLoss()

def batch_infonce_loss(z1, z2, temperature=0.15):
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    sim = torch.mm(z1, z2.T) / temperature
    labels = torch.arange(z1.shape[0], device=z1.device)
    return (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2.0
train_mask_tensor = torch.tensor(train_actor_mask, dtype=torch.bool, device=device)
train_labels_tensor = torch.tensor(actor_train_labels, dtype=torch.float32, device=device)
print("Starting Multi-Task GNN Training (20 Epochs)...")
for epoch in range(1, 21):
    model.train()
    total_fraud_loss = 0.0
    total_edge_loss = 0.0
    total_infonce_loss = 0.0
    batch_count = 0
    for (batch_seeds,) in train_seeds_loader:
        x_sub, edge_idx_sub, edge_attr_sub, n_targets = fetch_subgraph(batch_seeds)
        if edge_idx_sub.shape[1] == 0:
            continue
        x_sub = x_sub.to(device)
        edge_idx_sub = edge_idx_sub.to(device)
        edge_attr_sub = edge_attr_sub.to(device)
        seed_indices = batch_seeds.to(device)
        optimizer.zero_grad()

        # 1. Forward Pass on Subgraph
        h_sub = model(x_sub, edge_idx_sub, edge_attr_sub)
        h_seeds = h_sub[:n_targets]
        # 2. Supervised Actor Fraud Head Loss (ONLY on training actors!)
        seed_train_mask = train_mask_tensor[seed_indices]
        if seed_train_mask.sum() > 0:
            fraud_logits = model.predict_actor_fraud(h_seeds[seed_train_mask])
            target_labels = train_labels_tensor[seed_indices[seed_train_mask]]
            loss_fraud = fraud_bce_loss_fn(fraud_logits, target_labels)
        else:
            loss_fraud = torch.tensor(0.0, device=device)
            
        # 3. Edge / Ring Link Affinity Loss (Positive vs Sampled Negative Edges)
        u_idx, v_idx = edge_idx_sub
        pos_edge_logits = model.predict_edge(h_sub[u_idx], h_sub[v_idx], edge_attr_sub)
        pos_edge_targets = torch.ones_like(pos_edge_logits)
        # Sample random negative edges
        neg_u = torch.randint(0, h_sub.size(0), (len(u_idx),), device=device)
        neg_v = torch.randint(0, h_sub.size(0), (len(v_idx),), device=device)
        neg_edge_logits = model.predict_edge(h_sub[neg_u], h_sub[neg_v], edge_attr_sub)
        neg_edge_targets = torch.zeros_like(neg_edge_logits)
        loss_edge = (edge_bce_loss_fn(pos_edge_logits, pos_edge_targets) + 
                     edge_bce_loss_fn(neg_edge_logits, neg_edge_targets)) / 2.0
        # 4. InfoNCE Auxiliary Regularization Loss (Augmented Views)
        x_a, idx_a, att_a = augment_graph(x_sub, edge_idx_sub, edge_attr_sub)
        x_b, idx_b, att_b = augment_graph(x_sub, edge_idx_sub, edge_attr_sub)
        z_a = model.project(model(x_a, idx_a, att_a)[:n_targets])
        z_b = model.project(model(x_b, idx_b, att_b)[:n_targets])
        loss_infonce = batch_infonce_loss(z_a, z_b)
        # Joint Multi-Task Loss
        loss = loss_fraud + 0.5 * loss_edge + 0.2 * loss_infonce
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_fraud_loss += loss_fraud.item()
        total_edge_loss += loss_edge.item()
        total_infonce_loss += loss_infonce.item()
        batch_count += 1
    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d} | Fraud BCE: {total_fraud_loss/batch_count:.4f} | Edge Loss: {total_edge_loss/batch_count:.4f} | InfoNCE: {total_infonce_loss/batch_count:.4f}")


Training Multi-Task GNN on: cuda
Starting Multi-Task GNN Training (20 Epochs)...
Epoch 01 | Fraud BCE: 1.3548 | Edge Loss: 0.5692 | InfoNCE: 5.5350
Epoch 02 | Fraud BCE: 1.3208 | Edge Loss: 0.2659 | InfoNCE: 4.8554
Epoch 04 | Fraud BCE: 1.3085 | Edge Loss: 0.1616 | InfoNCE: 4.2388
Epoch 06 | Fraud BCE: 1.3125 | Edge Loss: 0.1265 | InfoNCE: 3.9505
Epoch 08 | Fraud BCE: 1.3164 | Edge Loss: 0.1166 | InfoNCE: 3.7892
Epoch 10 | Fraud BCE: 1.3060 | Edge Loss: 0.1068 | InfoNCE: 3.6864
Epoch 12 | Fraud BCE: 1.3184 | Edge Loss: 0.1082 | InfoNCE: 3.5931
Epoch 14 | Fraud BCE: 1.3100 | Edge Loss: 0.1022 | InfoNCE: 3.5290
Epoch 16 | Fraud BCE: 1.3120 | Edge Loss: 0.0997 | InfoNCE: 3.4404
Epoch 18 | Fraud BCE: 1.3080 | Edge Loss: 0.0948 | InfoNCE: 3.3927
Epoch 20 | Fraud BCE: 1.3161 | Edge Loss: 0.0971 | InfoNCE: 3.3459


In [126]:
# ============================================================
# EXTRACT TRAINED GNN EMBEDDINGS & CALIBRATED GRAPH SCORES
# ============================================================
model.eval()
with torch.no_grad():
    h_all = model(
        x_tensor.to(device),
        edge_index_tensor.to(device),
        edge_attr_tensor.to(device)
    )
    
    # 1. Calibrated Actor Fraud Score (Sigmoid of trained fraud head)
    actor_fraud_scores = torch.sigmoid(model.predict_actor_fraud(h_all)).cpu().numpy()
    
    # 2. 32-Dimensional Node Embeddings
    actor_embeddings = h_all.cpu().numpy()
    
    # 3. Calibrated Ring / Cluster Cohesion Score
    u_idx, v_idx = edge_index_tensor.to(device)
    edge_affinities = torch.sigmoid(model.predict_edge(h_all[u_idx], h_all[v_idx], edge_attr_tensor.to(device)))
    
    # Scatter mean edge affinity per actor node
    actor_ring_affinity = torch.zeros(h_all.size(0), device=device)
    actor_edge_count = torch.zeros(h_all.size(0), device=device)
    actor_ring_affinity.index_add_(0, u_idx, edge_affinities)
    actor_edge_count.index_add_(0, u_idx, torch.ones_like(edge_affinities))
    
    ring_scores = (actor_ring_affinity / torch.clamp(actor_edge_count, min=1.0)).cpu().numpy()

emb_cols = [f"actor_emb_{i}" for i in range(actor_embeddings.shape[1])]
actor_embedding_df = pd.DataFrame(actor_embeddings, columns=emb_cols)
actor_embedding_df["ActorID"] = actors
actor_embedding_df["actor_fraud_score"] = actor_fraud_scores
actor_embedding_df["actor_ring_score"]  = ring_scores

# Merge onto train and validation sets
train_clean_gnn = train_enhanced.merge(
    actor_embedding_df, on="ActorID", how="left"
).fillna(0)
val_clean_gnn = val_enhanced.merge(
    actor_embedding_df, on="ActorID", how="left"
).fillna(0)

print("Actor embeddings & Trained scores attached cleanly!")
print(f"Train shape: {train_clean_gnn.shape} | Val shape: {val_clean_gnn.shape}")
print("Mean Actor Fraud Score:", train_clean_gnn['actor_fraud_score'].mean())
print("Mean Actor Ring Score:", train_clean_gnn['actor_ring_score'].mean())

Actor embeddings & Trained scores attached cleanly!
Train shape: (472432, 107) | Val shape: (118108, 107)
Mean Actor Fraud Score: 0.50118476
Mean Actor Ring Score: 0.16154046


In [72]:
print(graph_features_df.columns.tolist())

['ActorID', 'actor_degree', 'mean_rarity', 'max_rarity', 'mean_shared_types', 'max_shared_types', 'hop_to_fraud', 'is_direct_accomplice', 'wcc_id', 'wcc_size', 'community', 'community_size', 'userCount', 'flaggedCount', 'flaggedPercent', 'unflagged_connected_to_flagged', 'card1']


In [127]:
# ============================================================
# 1. DERIVED TOPOLOGICAL FEATURES & HYBRID FEATURE SET
# ============================================================
graph_clean_features = [
    "actor_degree",
    "mean_rarity",
    "max_rarity",
    "mean_shared_types",
    "max_shared_types",
    "wcc_size",
    "community_size",
    "actor_fraud_score",   # From trained GNN fraud head
    "actor_ring_score",    # From trained GNN edge/ring head
]
# Top GNN embedding dimensions
gnn_emb_cols = [f"actor_emb_{i}" for i in range(8)]
# Derived topological ratios
for df_set in [train_clean_gnn, val_clean_gnn]:
    df_set["degree_to_community_ratio"] = df_set["actor_degree"] / (df_set["community_size"] + 1)
    df_set["rarity_density"] = df_set["mean_rarity"] * df_set["mean_shared_types"]
CLEAN_HYBRID_FEATURES = (
    TABULAR_BASE_FEATURES
    + graph_clean_features
    + ["degree_to_community_ratio", "rarity_density"]
    + gnn_emb_cols
)
X_train_final = train_clean_gnn[CLEAN_HYBRID_FEATURES].fillna(-999).copy()
X_val_final   = val_clean_gnn[CLEAN_HYBRID_FEATURES].fillna(-999).copy()
# Categorical encoding
categorical_cols = ["ProductCD", "card6", "DeviceInfo", "DeviceType", "id_31", "id_33", "P_emaildomain"]
for col in categorical_cols:
    combined = pd.concat([X_train_final[col], X_val_final[col]]).fillna("__MISSING__").astype(str)
    categories = pd.Index(combined.unique())
    mapping = {val: idx for idx, val in enumerate(categories)}
    X_train_final[col] = X_train_final[col].fillna("__MISSING__").astype(str).map(mapping).astype("int32")
    X_val_final[col]   = X_val_final[col].fillna("__MISSING__").astype(str).map(mapping).fillna(-1).astype("int32")
clean_xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)
clean_xgb.fit(
    X_train_final, y_train,
    eval_set=[(X_val_final, y_val)],
    verbose=100
)
clean_pred = clean_xgb.predict_proba(X_val_final)[:, 1]
print("\n============================================================")
print("       MODEL 3: CLEAN HYBRID GNN BENCHMARK                  ")
print("============================================================")
print(f"ROC-AUC: {roc_auc_score(y_val, clean_pred):.4f}")
print(f"PR-AUC : {average_precision_score(y_val, clean_pred):.4f}")

[0]	validation_0-aucpr:0.34646
[100]	validation_0-aucpr:0.44082
[200]	validation_0-aucpr:0.45073
[300]	validation_0-aucpr:0.45255
[400]	validation_0-aucpr:0.46297
[499]	validation_0-aucpr:0.47409

       MODEL 3: CLEAN HYBRID GNN BENCHMARK                  
ROC-AUC: 0.8876
PR-AUC : 0.4742


In [ ]:
# ============================================================
# IDENTIFY PREDICTED ABUSE RINGS IN VALIDATION DATA
# ============================================================

val_clean_gnn["risk_score"] = clean_pred
# Attach actual community ID if not present
if "community" not in val_clean_gnn.columns:
    actor_to_comm = graph_features_df.set_index("ActorID")["community"].to_dict()
    val_clean_gnn["community"] = val_clean_gnn["ActorID"].map(actor_to_comm).fillna(-1).astype(int)
# Filter top 1% riskiest transactions
top_threshold = np.percentile(clean_pred, 99)
flagged_tx = val_clean_gnn[val_clean_gnn["risk_score"] >= top_threshold].copy()
# Focus on multi-actor syndicates (rings with >= 2 members)
flagged_rings = flagged_tx[flagged_tx["community_size"] > 1]
# Group by the ACTUAL Community / Ring ID (not community_size!)
ring_summary = (
    flagged_rings.groupby("community")
    .agg(
        ring_members=("community_size", "first"),
        flagged_tx_count=("TransactionID", "count"),
        actual_frauds_caught=("isFraud", "sum"),
        avg_risk_score=("risk_score", "mean"),
        avg_gnn_ring_score=("actor_ring_score", "mean"),
        unique_actors=("ActorID", "nunique"),
        unique_cards=("card1", "nunique"),
        unique_devices=("DeviceInfo", "nunique")
    )
    .sort_values(["actual_frauds_caught", "flagged_tx_count"], ascending=False)
)
print(f"Total Unique Syndicate Rings Flagged in Validation: {len(ring_summary):,}")
print("\nTop Coordinated Abuse Rings Caught in Validation:")
display(ring_summary.head(10))

Total Unique Syndicate Rings Flagged in Validation: 11

Top Coordinated Abuse Rings Caught in Validation:


,ring_members,flagged_tx_count,actual_frauds_caught,avg_risk_score,avg_gnn_ring_score,unique_actors,unique_cards,unique_devices
community,,,,,,,,
4885,844.0,722,625,0.901366,0.963749,135,81,77
6120,477.0,34,28,0.874818,0.492142,19,16,8
10939,95.0,7,7,0.935639,0.824386,3,3,5
253,113.0,2,2,0.856659,0.500000,2,2,2
8737,2.0,2,2,0.903557,0.000003,1,1,2
24172,134.0,2,2,0.976861,0.732140,2,2,2
13393,3.0,1,1,0.851905,0.330913,1,1,1
13936,3.0,1,1,0.965285,0.478567,1,1,1
19469,122.0,1,1,0.950778,0.951183,1,1,1


In [ ]:
# import torch
# print(torch.__version__)
# print(torch.version.cuda)

2.14.0+cu126
12.6


In [129]:
# ============================================================
# MERCHANT POLICY ENGINE: COST-SENSITIVE RING MITIGATION
# ============================================================
import numpy as np
import pandas as pd

def evaluate_merchant_cost(
    y_true, 
    y_prob, 
    amounts, 
    chargeback_fee=20.0, 
    margin=0.15, 
    ltv_loss_rate=0.05
):
    thresholds = np.linspace(0.01, 0.99, 100)
    results = []

    for t in thresholds:
        declined = y_prob >= t
        
        # Fraud outcomes
        tp = (declined) & (y_true == 1)      # Stopped Fraud
        fn = (~declined) & (y_true == 1)     # Missed Fraud (Loss)
        
        # Benign customer outcomes
        fp = (declined) & (y_true == 0)      # Insulted Good Customer (Friction)
        tn = (~declined) & (y_true == 0)     # Clean Sale
        
        # Dollar impacts
        fraud_loss_prevented = amounts[tp].sum()
        chargeback_fees_saved = tp.sum() * chargeback_fee
        
        fraud_losses_incurred = amounts[fn].sum() + (fn.sum() * chargeback_fee)
        
        # False Positive Cost: Lost margin + long-term customer churn
        fp_cost = (amounts[fp] * margin).sum() + (amounts[fp] * ltv_loss_rate).sum()
        
        net_merchant_loss = fraud_losses_incurred + fp_cost
        
        results.append({
            "threshold": t,
            "net_loss": net_merchant_loss,
            "fraud_prevented_val": fraud_loss_prevented,
            "fp_cost": fp_cost,
            "tp_count": tp.sum(),
            "fp_count": fp.sum(),
            "fn_count": fn.sum(),
            "precision": tp.sum() / max(1, (tp.sum() + fp.sum())),
            "recall": tp.sum() / max(1, (tp.sum() + fn.sum()))
        })

    df_res = pd.DataFrame(results)
    best_idx = df_res["net_loss"].idxmin()
    return df_res.loc[best_idx], df_res

# Evaluate on Validation Set
amounts_val = val_clean_gnn["TransactionAmt"].values
best_policy, cost_curve = evaluate_merchant_cost(
    y_true=y_val.values, 
    y_prob=clean_pred, 
    amounts=amounts_val
)

print("========== MERCHANT COST-OPTIMAL POLICY ==========")
print(f"Optimal Decision Threshold:  {best_policy['threshold']:.3f}")
print(f"Stopped Fraud Volume:        ${best_policy['fraud_prevented_val']:,.2f}")
print(f"False Positive Friction Cost: ${best_policy['fp_cost']:,.2f}")
print(f"Optimal Operating Recall:    {best_policy['recall']:.2%}")
print(f"Optimal Operating Precision: {best_policy['precision']:.2%}")

========== MERCHANT COST-OPTIMAL POLICY ==========
Optimal Decision Threshold:  0.297
Stopped Fraud Volume:        $229,415.54
False Positive Friction Cost: $60,191.02
Optimal Operating Recall:    44.54%
Optimal Operating Precision: 53.50%


In [130]:
# ============================================================
# CONTINUOUS STREAMING VELOCITY SPIKE DETECTOR (CAUSAL TIMELINE)
# ============================================================
import pandas as pd
import numpy as np
def compute_continuous_streaming_velocity(train_df, val_df, time_col="TransactionDT"):
    """
    Computes streaming burst counters across a continuous chronological timeline.
    State accumulated in train flows seamlessly into validation without reset!
    Zero future leakage: strictly backward-looking historical windows.
    """
    train_work = train_df[[time_col, "card1", "DeviceInfo"]].copy()
    train_work["is_train"] = True
    train_work["orig_order"] = np.arange(len(train_work))
    
    val_work = val_df[[time_col, "card1", "DeviceInfo"]].copy()
    val_work["is_train"] = False
    val_work["orig_order"] = np.arange(len(val_work))
    combined = pd.concat([train_work, val_work], ignore_index=True)
    combined["orig_idx"] = np.arange(len(combined))
    combined["dt_sec"] = pd.to_datetime(combined[time_col], unit="s")
    
    # Sort chronologically across the entire continuum
    combined = combined.sort_values("dt_sec")
    
    combined["card1_clean"] = combined["card1"].fillna(-1).astype("int64")
    combined["device_clean"] = combined["DeviceInfo"].fillna("__UNKNOWN__").astype(str)
    # 1. 5-minute rolling card velocity (strictly past transactions: count - 1)
    card_5m = (
        combined.set_index("dt_sec")
        .groupby("card1_clean")["orig_idx"]
        .rolling("300s")
        .count() - 1
    ).reset_index()
    card_5m.rename(columns={"orig_idx": "card_velocity_5m"}, inplace=True)
    # 2. 24-hour rolling card velocity
    card_24h = (
        combined.set_index("dt_sec")
        .groupby("card1_clean")["orig_idx"]
        .rolling("86400s")
        .count() - 1
    ).reset_index()
    card_24h.rename(columns={"orig_idx": "card_velocity_24h"}, inplace=True)
    # 3. 30-minute rolling device velocity
    dev_30m = (
        combined.set_index("dt_sec")
        .groupby("device_clean")["orig_idx"]
        .rolling("1800s")
        .count() - 1
    ).reset_index()
    dev_30m.rename(columns={"orig_idx": "device_velocity_30m"}, inplace=True)
    # Assign aligned values
    combined["card_velocity_5m"]    = card_5m["card_velocity_5m"].values
    combined["card_velocity_24h"]   = card_24h["card_velocity_24h"].values
    combined["device_velocity_30m"] = dev_30m["device_velocity_30m"].values
    # 4. Spike Ratio: Short-term (5m) burst relative to long-term (24h) baseline
    combined["card_spike_ratio"] = (
        (combined["card_velocity_5m"] + 1.0) /
        ((combined["card_velocity_24h"] / 288.0) + 1.0)
    )
    # Split back into train and val preserving exact original row order
    train_res = combined[combined["is_train"]].sort_values("orig_order")
    val_res   = combined[~combined["is_train"]].sort_values("orig_order")
    stream_cols = ["card_velocity_5m", "card_velocity_24h", "device_velocity_30m", "card_spike_ratio"]
    
    train_out = train_df.copy()
    val_out   = val_df.copy()
    for col in stream_cols:
        train_out[col] = train_res[col].values
        val_out[col]   = val_res[col].values
    return train_out, val_out
print("Computing continuous streaming velocity across train -> val continuum...")
train_stream, val_stream = compute_continuous_streaming_velocity(train_clean_gnn, val_clean_gnn)
print("Continuous streaming velocity computed successfully!")
print("Val 24h card velocity mean (retains history):", val_stream["card_velocity_24h"].mean())
print("Val stream shape:  ", val_stream.shape)

Computing continuous streaming velocity across train -> val continuum...
Continuous streaming velocity computed successfully!
Val 24h card velocity mean (retains history): 21.612600331899618
Val stream shape:   (118108, 115)


In [ ]:
train_stream.head()
val_stream.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,gnn_emb_61,gnn_emb_62,gnn_emb_63,degree_to_community_ratio,rarity_density,risk_score,card_velocity_5m,card_velocity_24h,device_velocity_30m,card_spike_ratio
0,3459432,1,12192900,33.261,C,9300,103.0,185.0,visa,138.0,...,0.000000,14164.432617,0.0,1.119078,22.460077,0.576110,0.0,0.0,0.0,1.00000
1,3459433,0,12192911,52.811,C,8809,179.0,106.0,visa,137.0,...,0.000000,37598.675781,0.0,0.670294,17.320619,0.095069,1.0,1.0,0.0,1.99308
2,3459434,0,12192913,136.956,C,10819,555.0,185.0,visa,226.0,...,0.075769,0.000000,0.0,0.000000,0.000000,0.322447,0.0,0.0,0.0,1.00000
3,3459435,0,12193040,136.956,C,9633,130.0,185.0,visa,138.0,...,0.000000,2705.557129,0.0,1.325864,24.438838,0.768930,0.0,0.0,0.0,1.00000
4,3459436,0,12193199,25.000,H,17188,321.0,150.0,visa,226.0,...,0.000000,0.000000,0.0,1.017427,20.319259,0.028799,0.0,0.0,0.0,1.00000


In [79]:
STREAM_FEATURES = [
    "card_velocity_5m",
    "card_velocity_24h",
    "device_velocity_30m",
    "card_spike_ratio"
]

# Fill missing/NaN
train_stream[STREAM_FEATURES] = train_stream[STREAM_FEATURES].fillna(0)
val_stream[STREAM_FEATURES]   = val_stream[STREAM_FEATURES].fillna(0)

In [132]:
train_stream[STREAM_FEATURES].isna().sum()

card_velocity_5m       0
card_velocity_24h      0
device_velocity_30m    0
card_spike_ratio       0
dtype: int64

In [139]:
a
from sklearn.metrics import roc_auc_score, average_precision_score

# ------------------------------------------------------------
# 0. CLEAN UP ANY SUFFIX ARTIFACTS FROM PREVIOUS RUNS
# ------------------------------------------------------------
for s in [train_stream, val_stream]:
    dup_cols = [c for c in s.columns if c.startswith("card1_amt_mean") or c.startswith("amt_to_mean_card1")]
    if dup_cols:
        s.drop(columns=dup_cols, inplace=True, errors="ignore")

# ------------------------------------------------------------
# 1. HIGH-SIGNAL BEHAVIORAL & TIME DECOMPOSITION (IN-PLACE)
# ------------------------------------------------------------
for split in [train_stream, val_stream]:
    split["hour"] = (split["TransactionDT"] // 3600) % 24
    split["day"]  = (split["TransactionDT"] // 86400) % 7
    split["amt_cents"] = split["TransactionAmt"] - np.floor(split["TransactionAmt"])
    split["amt_log"]   = np.log1p(split["TransactionAmt"])

# ------------------------------------------------------------
# 2. CAUSAL CARD HISTORICAL AVERAGE SPEND (VIA MAP - ZERO MERGE CONFLICTS)
# ------------------------------------------------------------
card1_means = train_stream.groupby("card1")["TransactionAmt"].mean().to_dict()

for split in [train_stream, val_stream]:
    split["card1_amt_mean"] = split["card1"].map(card1_means).fillna(split["TransactionAmt"])
    split["amt_to_mean_card1"] = split["TransactionAmt"] / (split["card1_amt_mean"] + 1e-5)

# ------------------------------------------------------------
# 3. DEFINITIVE CHAMPION FEATURE SET (WITH GENUINE GNN SIGNALS)
# ------------------------------------------------------------
gnn_emb_subset = [f"actor_emb_{i}" for i in range(8)]

CHAMPION_FEATURES = [
    # 1. Amounts & Behavioral Ratios
    "TransactionAmt", "amt_log", "amt_cents", "amt_to_mean_card1", "hour", "day",
    
    # 2. Categorical Identity & Hardware
    "ProductCD", "card1", "card2", "card3", "card5", "card6",
    "addr1", "addr2", "P_emaildomain", "DeviceInfo",
    
    # 3. Standard IEEE-CIS Counting & Timedelta Signals
    "C1", "C2", "C13", "C14", "D1", "D2", "D15",
    
    # 4. Continuous Real-Time Streaming Velocity
    "card_velocity_5m", "card_velocity_24h", "device_velocity_30m", "card_spike_ratio",
    
    # 5. Pure Graph Structural Topology
    "actor_degree", "wcc_size", "community_size", "mean_rarity",
    
    # 6. Genuine Trained GNN Outputs (Supervised Fraud Score + Ring Score + Embeddings)
    "actor_fraud_score",
    "actor_ring_score",
] + gnn_emb_subset

# Categorical mapping
cat_cols = ["ProductCD", "card6", "P_emaildomain", "DeviceInfo"]
for col in cat_cols:
    train_vals = train_stream[col].fillna("__MISSING__").astype(str)
    categories = pd.Index(train_vals.unique())
    mapping = {val: idx for idx, val in enumerate(categories)}
    train_stream[col] = train_vals.map(mapping).astype("int32")
    val_stream[col]   = val_stream[col].fillna("__MISSING__").astype(str).map(mapping).fillna(-1).astype("int32")

# Prepare clean training matrices
X_train_champ = train_stream[CHAMPION_FEATURES].fillna(-999).copy()
X_val_champ   = val_stream[CHAMPION_FEATURES].fillna(-999).copy()
y_train_champ = train_stream["isFraud"].astype(int)
y_val_champ   = val_stream["isFraud"].astype(int)

print(f"X_train_champ: {X_train_champ.shape} ({len(CHAMPION_FEATURES)} features)")
print(f"X_val_champ:   {X_val_champ.shape}")

# ------------------------------------------------------------
# 4. FIT CHAMPION SENTINEL BOOSTER (HYPERPARAMETER TUNING)
# ------------------------------------------------------------
champion_xgb = XGBClassifier(
    n_estimators=700,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

champion_xgb.fit(
    X_train_champ,
    y_train_champ,
    eval_set=[(X_val_champ, y_val_champ)],
    verbose=100
)

# ------------------------------------------------------------
# 5. BENCHMARK EVALUATION
# ------------------------------------------------------------
champ_pred = champion_xgb.predict_proba(X_val_champ)[:, 1]
final_roc = roc_auc_score(y_val_champ, champ_pred)
final_pr  = average_precision_score(y_val_champ, champ_pred)

print("\n============================================================")
print("          CHAMPION SENTINEL MODEL BENCHMARK                 ")
print("============================================================")
print(f"ROC-AUC: {final_roc:.4f}")
print(f"PR-AUC : {final_pr:.4f}")



X_train_champ: (472432, 41) (41 features)
X_val_champ:   (118108, 41)
[0]	validation_0-aucpr:0.37404
[100]	validation_0-aucpr:0.46355
[200]	validation_0-aucpr:0.47108
[300]	validation_0-aucpr:0.47520
[400]	validation_0-aucpr:0.47920
[500]	validation_0-aucpr:0.47549
[600]	validation_0-aucpr:0.46955
[699]	validation_0-aucpr:0.47406

          CHAMPION SENTINEL MODEL BENCHMARK                 
ROC-AUC: 0.9005
PR-AUC : 0.4741


In [140]:

champion_xgb.save_model("sentinel_xgb.json")
print("\n[SUCCESS] Champion booster exported to sentinel_xgb.json!")


[SUCCESS] Champion booster exported to sentinel_xgb.json!
